In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# 1. Load and clean
df = pd.read_csv("uber.csv")
df = df.drop(columns=['Unnamed: 0', 'key'], errors='ignore').dropna()

df = df[
    (df['fare_amount'] > 0) &
    df['passenger_count'].between(1, 6) &
    df['pickup_latitude'].between(-90, 90) &
    df['dropoff_latitude'].between(-90, 90) &
    df['pickup_longitude'].between(-180, 180) &
    df['dropoff_longitude'].between(-180, 180)
].copy()

# 2. Date features
dt = pd.to_datetime(df['pickup_datetime'], errors='coerce', utc=True)
df['hour'] = dt.dt.hour
df['month'] = dt.dt.month
df['year'] = dt.dt.year
df['dayofweek'] = dt.dt.dayofweek
df = df.drop(columns='pickup_datetime').dropna()

# 3. Distance (Haversine)
lat1, lon1, lat2, lon2 = map(np.radians, [
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
])
a = np.sin((lat2 - lat1) / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2
df['distance_km'] = 6371 * 2 * np.arcsin(np.sqrt(a))
df = df[df['distance_km'].between(1, 130)]

# 4. Sample for speed, then split
df = df.sample(n=min(50000, len(df)), random_state=42)
X = df.drop(columns='fare_amount')
y = df['fare_amount']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# 5. Train and evaluate both models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=30, max_depth=15, n_jobs=-1, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"{name}: R2 = {r2_score(y_test, pred):.3f}, "
          f"RMSE = {np.sqrt(mean_squared_error(y_test, pred)):.3f}")

: 